In [5]:
# Blinkit Business Analysis

In [6]:
import pandas as pd

In [7]:
## importing data
orders = pd.read_csv('../data/blinkit_orders.csv')
products = pd.read_csv('../data/blinkit_products.csv')
order_items = pd.read_csv('../data/blinkit_order_items.csv')
customers = pd.read_csv('../data/blinkit_customers.csv')
customer_feedback = pd.read_csv('../data/blinkit_customer_feedback.csv')

In [8]:
## merging the tables
sales_data = orders.merge(order_items, on="order_id")
sales_data = sales_data.merge(products, on="product_id")
sales_data = sales_data.merge(customers, on="customer_id")
sales_data = sales_data.merge(customer_feedback,
    on=['customer_id', 'order_id'],
    how='left')

In [9]:
sales_data.shape

(5000, 38)

In [10]:
## data cleaning
sales_data.isnull().sum()

order_id                  0
customer_id               0
order_date                0
promised_delivery_time    0
actual_delivery_time      0
delivery_status           0
order_total               0
payment_method            0
delivery_partner_id       0
store_id                  0
product_id                0
quantity                  0
unit_price                0
product_name              0
category                  0
brand                     0
price                     0
mrp                       0
margin_percentage         0
shelf_life_days           0
min_stock_level           0
max_stock_level           0
customer_name             0
email                     0
phone                     0
address                   0
area                      0
pincode                   0
registration_date         0
customer_segment          0
total_orders              0
avg_order_value           0
feedback_id               0
rating                    0
feedback_text             0
feedback_category   

In [11]:
sales_data.duplicated().sum()

np.int64(0)

In [12]:
## Fixing the data format
sales_data['order_date'] = pd.to_datetime(sales_data['order_date'])
sales_data['promised_delivery_time'] = pd.to_datetime(sales_data['promised_delivery_time'])
sales_data['actual_delivery_time'] = pd.to_datetime(sales_data['actual_delivery_time'])

C:\Users\megha\AppData\Local\Temp\ipykernel_16944\939641917.py:2: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sales_data['order_date'] = pd.to_datetime(sales_data['order_date'])
C:\Users\megha\AppData\Local\Temp\ipykernel_16944\939641917.py:3: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sales_data['promised_delivery_time'] = pd.to_datetime(sales_data['promised_delivery_time'])
C:\Users\megha\AppData\Local\Temp\ipykernel_16944\939641917.py:4: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  sales_data['actual_delivery_time'] = pd.to_datetime(sales_data['actual_delivery_time'])


In [13]:
sales_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_id                5000 non-null   int64         
 1   customer_id             5000 non-null   int64         
 2   order_date              5000 non-null   datetime64[us]
 3   promised_delivery_time  5000 non-null   datetime64[us]
 4   actual_delivery_time    5000 non-null   datetime64[us]
 5   delivery_status         5000 non-null   str           
 6   order_total             5000 non-null   float64       
 7   payment_method          5000 non-null   str           
 8   delivery_partner_id     5000 non-null   int64         
 9   store_id                5000 non-null   int64         
 10  product_id              5000 non-null   int64         
 11  quantity                5000 non-null   int64         
 12  unit_price              5000 non-null   float64       
 13 

In [14]:
## creating revenue column
sales_data['revenue'] = sales_data['quantity'] * sales_data['price']
sales_data['order_month'] = sales_data['order_date'].dt.to_period('M')
sales_data['order_day'] = sales_data['order_date'].dt.day_name()

In [15]:
sales_data['revenue'].sum()

np.float64(4972415.43)

In [16]:
# Top selling Products by Quantity
sales_data.groupby("product_name")["quantity"].sum().sort_values(ascending=False).head(5)

product_name
Pet Treats        473
Toilet Cleaner    430
Dish Soap         397
Vitamins          380
Cough Syrup       373
Name: quantity, dtype: int64

In [17]:
# Top selling Products by Revenue
sales_data.groupby("product_name")["revenue"].sum().sort_values(ascending=False).head(5)

product_name
Vitamins          260822.01
Pet Treats        252007.37
Cough Syrup       203569.98
Toilet Cleaner    199837.48
Bread             184851.10
Name: revenue, dtype: float64

In [18]:
sales_data['order_date'].min() 

Timestamp('2023-03-16 08:10:00')

In [19]:
sales_data['order_date'].max() 

Timestamp('2024-11-04 20:29:00')

In [ ]:
## analysing revenue trend on monthly basis
monthly_sales = sales_data.groupby('order_month')['revenue'].sum()
monthly_sales.sort_values(ascending=False)

order_month
2023-08    303511.99
2023-09    277585.47
2024-01    273090.91
2023-05    269445.26
2024-05    264996.72
2023-10    263859.88
2023-11    261474.62
2024-02    260130.49
2024-10    255862.84
2023-12    253628.38
2024-07    252367.96
2024-08    251255.76
2023-04    249382.54
2024-03    240661.78
2024-06    239157.46
2023-07    237060.98
2023-06    233672.95
2024-09    231024.68
2024-04    215255.49
2023-03    110989.65
2024-11     27999.62
Freq: M, Name: revenue, dtype: float64

In [ ]:
## Revenue remained relatively stable across the months , 
# eventhoug comparing to all the months  in august 2023 the revenue was huge and 
# in april 2024 month revenue was less comparing to all other months

In [21]:
## august 2023 is the least revenue generated month so analysing which product makes more revenue
sales_data[sales_data['order_month'] == '2023-08'] \
.groupby('product_name')['revenue'].sum().sort_values(ascending=False).head(10)

product_name
Onions            15234.87
Carrots           13647.58
Cat Food          11753.77
Toilet Cleaner    11621.93
Iced Tea          11549.61
Dish Soap         11383.43
Pain Reliever     11331.60
Pet Treats        11303.52
Soap              10978.86
Bread             10283.91
Name: revenue, dtype: float64

In [22]:
## analysing which category is responsible for the high revenue in april 2024
sales_data[sales_data['order_month'] == '2023-08'] \
.groupby('category')['revenue'].sum().sort_values(ascending=False)

category
Fruits & Vegetables      46037.39
Dairy & Breakfast        38429.83
Cold Drinks & Juices     37912.12
Pet Care                 29254.89
Personal Care            29106.69
Pharmacy                 28015.96
Household Care           27585.97
Snacks & Munchies        21786.02
Instant & Frozen Food    18017.47
Grocery & Staples        13852.37
Baby Care                13513.28
Name: revenue, dtype: float64

In [23]:
## april 2024 is the least revenue generated month so analysing which product makes less revenue
sales_data[sales_data['order_month'] == '2024-04'] \
.groupby('product_name')['revenue'].sum().sort_values(ascending=True).head(10)

product_name
Mango Drink     321.28
Spinach         630.84
Chocolates     1156.74
Milk           1197.66
Rice           1251.06
Shampoo        1270.68
Cereal         1423.11
Carrots        1696.43
Tomatoes       1779.05
Toothpaste     1798.56
Name: revenue, dtype: float64

In [24]:
## analysing which category is responsible for the less revenue in april 2024
sales_data[sales_data['order_month'] == '2024-04'] \
.groupby('category')['revenue'].sum().sort_values(ascending=True)

category
Instant & Frozen Food    12149.34
Cold Drinks & Juices     13823.29
Fruits & Vegetables      15258.79
Personal Care            16389.77
Grocery & Staples        17242.19
Baby Care                19316.77
Household Care           19535.37
Snacks & Munchies        21438.43
Pharmacy                 23441.78
Pet Care                 23757.11
Dairy & Breakfast        32902.65
Name: revenue, dtype: float64

In [25]:
# analysing day by day revenue
sales_data.groupby('order_day')['revenue'].sum().sort_values(ascending=False)

order_day
Wednesday    753018.67
Tuesday      735500.53
Thursday     716915.86
Saturday     700107.86
Sunday       694406.86
Monday       693907.44
Friday       678558.21
Name: revenue, dtype: float64

In [ ]:
## weekend and weekdays performence showed minimal variation

In [27]:
sales_data['day_type'] = sales_data['order_day'].apply(
    lambda x: 'Weekend' if x in ['Saturday', 'Sunday'] else 'Weekday'
)

sales_data.groupby('day_type')['revenue'].mean()

day_type
Weekday    1001.932431
Weekend     975.867544
Name: revenue, dtype: float64

In [ ]:
## the weekend performence are nearly equal to the overall weekday performence

In [29]:
## top 10 customers on revenue basis 
sales_data.groupby(
    ['customer_id', 'customer_name', 'area']
)['revenue'].sum().sort_values(ascending=False).head()

customer_id  customer_name  area          
25128143     Odika Kannan   Mahbubnagar       10533.39
77869660     Nidhi Sha      Gandhinagar       10115.75
12272282     Dev Bal        Sikar              9924.84
4597433      Lipika Kumer   Tadepalligudem     9553.69
33331259     Leena Deol     Saharsa            9403.77
Name: revenue, dtype: float64

In [30]:
sales_data.groupby(['customer_id' , 'customer_name', 'area'])['order_id'].nunique().sort_values(ascending=False).head()

customer_id  customer_name  area        
77869660     Nidhi Sha      Gandhinagar     9
17805991     Jhalak Rai     Dhule           8
8791577      Warda Kohli    Madhyamgram     8
93018527     Lekha Rout     Narasaraopet    7
12832151     Ekavir Bhalla  Baranagar       7
Name: order_id, dtype: int64

In [ ]:
## analysing the average order value
customer_orders = sales_data.groupby(['customer_id','customer_name','area'])['order_id'].nunique()
customer_revenue = sales_data.groupby(['customer_id', 'customer_name', 'area'])['revenue'].sum()

aov = customer_revenue / customer_orders
aov.sort_values(ascending=False)

customer_id  customer_name    area        
64489709     Ekavir Kade      North Dumdum    2929.65
91602727     Rayaan Palla     Indore          2906.82
41439103     Ati Wali         Junagadh        2877.03
81655250     Kashvi Rege      Madurai         2843.85
949062       Jackson Karpe    Mau             2840.58
                                               ...   
44859592     Dhriti Koshy     Proddatur         22.04
76441843     Neelima Chander  Gulbarga          16.86
81462774     Ishaan Sarkar    Junagadh          16.86
18940384     Anita Sehgal     Sonipat           13.25
89472859     Ekaraj Dash      Saharsa           12.32
Length: 2172, dtype: float64

In [32]:
## analysing repeated and new customers
customer_orders = sales_data.groupby('customer_id')['order_id'].nunique()

repeat_customers = (customer_orders > 1).sum()
one_time_customers = (customer_orders == 1).sum()

print("Repeat customers:", repeat_customers)
print("One-time customers:", one_time_customers)


Repeat customers: 1492
One-time customers: 680


In [33]:
## calculating the one time and repeated customers ordering in blinkit
total = repeat_customers + one_time_customers


repeat_percent = (repeat_customers / total) * 100
one_time_percent = (one_time_customers / total) * 100

print('repeated customer:' , repeat_percent ,'\n', 'one_time customer:',one_time_percent)


repeated customer: 68.69244935543279 
 one_time customer: 31.30755064456722


In [34]:
## analysing the revenue generated by repeated and one time customer
customer_orders = sales_data.groupby('customer_id')['order_id'].nunique()
customer_revenue = sales_data.groupby('customer_id')['revenue'].sum()

customer_type = pd.DataFrame({
    'orders': customer_orders,
    'revenue': customer_revenue
})

customer_type['type'] = customer_type['orders'].apply(lambda x: 'Repeat' if x > 1 else 'One-time')

customer_type.groupby('type')['revenue'].sum()

type
One-time     659314.99
Repeat      4313100.44
Name: revenue, dtype: float64

In [35]:
## converting revenue generated by customer type in percentage
revenue_by_type = customer_type.groupby('type')['revenue'].sum()

repeat_percent = (
    revenue_by_type['Repeat'] / revenue_by_type.sum()
) * 100

one_time_percent = (
    revenue_by_type['One-time'] / revenue_by_type.sum()
) * 100

print("Repeat customer revenue %:", repeat_percent)
print("One-time customer revenue %:", one_time_percent)

Repeat customer revenue %: 86.74054894886368
One-time customer revenue %: 13.259451051136326
